# 1396. Design Underground System

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** design, hash-table
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-underground-system/)

An underground railway system is keeping track of customer travel times between
different stations. They are using this data to calculate the average time it takes
to travel from one station to another.

Implement the `UndergroundSystem` class:

- `checkIn(id, stationName, t)` - a customer with card id `id` checks in at station
  `stationName` at time `t`. A customer can only be checked into one place at a time.
- `checkOut(id, stationName, t)` - a customer with card id `id` checks out from
  station `stationName` at time `t`.
- `getAverageTime(startStation, endStation)` - returns the average time it takes to
  travel from `startStation` to `endStation`. The average is computed from all the
  previous traveling times from `startStation` to `endStation` that happened
  **directly** - that is, a check in at `startStation` followed by a check out from
  `endStation`. The time it takes to travel from `startStation` to `endStation` may
  be different from the time it takes from `endStation` to `startStation`.

There will be at least one customer that has traveled from `startStation` to
`endStation` before `getAverageTime` is called.

---

### Example

```
undergroundSystem.checkIn(45, "Leyton", 3);
undergroundSystem.checkIn(32, "Paradise", 8);
undergroundSystem.checkIn(27, "Leyton", 10);
undergroundSystem.checkOut(45, "Waterloo", 15);      // 45: Leyton -> Waterloo in 12
undergroundSystem.checkOut(27, "Waterloo", 20);      // 27: Leyton -> Waterloo in 10
undergroundSystem.checkOut(32, "Cambridge", 22);     // 32: Paradise -> Cambridge in 14
undergroundSystem.getAverageTime("Paradise", "Cambridge");  // 14.0
undergroundSystem.getAverageTime("Leyton", "Waterloo");     // 11.0  = (12 + 10) / 2
undergroundSystem.checkIn(10, "Leyton", 24);
undergroundSystem.getAverageTime("Leyton", "Waterloo");     // 11.0  - 10 is still travelling
undergroundSystem.checkOut(10, "Waterloo", 38);      // 10: Leyton -> Waterloo in 14
undergroundSystem.getAverageTime("Leyton", "Waterloo");     // 12.0  = (12 + 10 + 14) / 3
```

---

### Constraints

- `1 <= id, t <= 10^6`
- `1 <= stationName.length, startStation.length, endStation.length <= 10`
- All times are consistent: a customer checks out **after** checking in
- At most `2 * 10^4` calls will be made in total
- Answers within `10^-5` of the actual value are accepted

This is Oyster, Navigo, Clipper - a real fare system, minus the fares. Two dicts that
mean two completely different things, and the interesting decision is what you store
in the second one.

## Before you write anything

**1.** Between `checkIn` and `checkOut` you must remember something about a
passenger. **What**, exactly - and for **how long**? Name the precise moment you are
allowed to forget it. Then say what happens to your memory if you never do: `2 * 10^4`
calls, a system that runs all day.

**2.** `getAverageTime` needs a mean. Two ways:

```
(a) keep every trip time in a list, and average it when asked
(b) keep a running (total, count) pair, and divide when asked
```

Price both, on **two** axes: the cost of one `getAverageTime` call, and the memory
after a million journeys on the same route. One of them is `O(1)` in both. Say why
you would ever choose the other anyway - what question can (a) answer that (b) cannot?

**3.** What is the key of the second dict? The route is a **pair** of stations. The
tempting key is `start + end`, or `start + "-" + end`. Give a pair of real station
names that makes string concatenation return the wrong answer, then say what a tuple
key does differently. (`("Kings", "Cross")` and `("Kings Cross", "")` - which one is
`"KingsCross"`?)

**4.** "The time from A to B may be different from B to A." So is your key **ordered**?
What does that mean for the number of entries in the dict on a network with 300
stations, worst case?

**5.** Nothing says a passenger travels once. Card `45` checks in, checks out, and
checks in again an hour later. Which of your two dicts changes, and which does not?

Now the subtle part, and read it twice. If `checkIn` **assigns**
(`self.travelling[id] = ...`), then forgetting to remove the entry on check-out is
*not* a wrong answer - the next check-in overwrites it, and every average stays
correct forever. It is a **leak**: the dict keeps one entry per passenger who ever
travelled, instead of one per passenger currently travelling. No test below catches
it. Nothing will tell you. You have to decide to care.

But write `self.travelling.setdefault(id, (stationName, t))` instead of an
assignment - a very natural thing to type - and the *combination* of that and the
missing removal measures journey two from journey one's check-in. That one the tests
do catch. Two bugs, one visible and one silent, from the same missing line.

**6.** How do you test it? `checkIn` and `checkOut` return nothing - only
`getAverageTime` speaks, and it speaks in floats. So a passenger left dangling in the
in-flight dict is invisible until an average is off by a fraction several calls later.
What would you check after every call to make that visible immediately?

## Two routes

**A - two dicts, one running total** *(write this first)*

```
self.travelling = {}    # id -> (station, time)          who is inside the system now
self.routes = {}        # (start, end) -> [total, count]  one entry per ordered pair
```

`checkIn` writes one entry. `checkOut` **removes** that entry (question 5) and folds
the duration into the route's `[total, count]`. `getAverageTime` is one division.
All three are `O(1)`, and memory is `O(passengers currently travelling + routes ever
used)` - never `O(journeys)`.

The tuple key `(start, end)` is doing real work: it is ordered, so `A -> B` and
`B -> A` are different entries automatically, and it cannot be confused by station
names containing whatever separator you would have picked.

**B - a dict of dicts**

```
self.routes = {start: {end: [total, count]}}
```

Identical costs, one extra lookup, and one new ability: *given a start station, what
are all the destinations and their averages?* - a question route A cannot answer
without scanning every key. That is a dashboard query, and it is the reason a real
system stores it this way.

> **Two dicts, two lifetimes.** `travelling` is short-lived and must be cleaned up on
> every check-out; `routes` is permanent and only ever grows by *one entry per new
> route*, never per journey. Keeping those two lifetimes straight is the whole design,
> and confusing them is how you build something that works for a day and dies in a week.

In [ ]:
class UndergroundSystem:

    def __init__(self):
        pass

    def checkIn(self, id: int, stationName: str, t: int) -> None:
        pass

    def checkOut(self, id: int, stationName: str, t: int) -> None:
        pass

    def getAverageTime(self, startStation: str, endStation: str) -> float:
        pass

### The test harness

Question 6's answer. Two of the three methods return `None`, so a passenger left
dangling in the in-flight dict, or a duration folded into the wrong route, is silent at
the moment you cause it.

So `check` replays a call sequence against your class **and** against a model that
keeps every individual journey time in a list per route. After **every** call it asks
your class for the average of *every route seen so far* and compares against the
model's mean, to within `10^-5`. That means a wrong total is reported at the call that
caused it, not several calls later - and it catches the classic bug where a re-used
card measures its second journey from its first check-in.

`stress` runs many passengers around a small network so cards are re-used constantly
and routes overlap.

**What it cannot see**, stated plainly: if `checkIn` assigns, a `checkOut` that never
removes the in-flight entry produces correct averages forever and only leaks memory.
Every test below passes. That is question 5, and the only defence is the
`len(self.travelling)` check in the section after the tests. Run this cell; don't
edit it.

In [ ]:
import random


def check(ops):
    '''Replay (op, args) against UndergroundSystem and a keep-every-journey model.'''
    log = []
    try:
        us = UndergroundSystem()
    except Exception as e:
        return False, [f"   !! UndergroundSystem() raised {type(e).__name__}: {e}"]

    inside, trips = {}, {}                    # id -> (station, t);  (a, b) -> [durations]

    for op, args in ops:
        call = f"{op}({', '.join(map(repr, args))})"
        try:
            if op == "checkIn":
                i, s, t = args
                us.checkIn(i, s, t)
                inside[i] = (s, t)
                log.append(call)
            elif op == "checkOut":
                i, s, t = args
                us.checkOut(i, s, t)
                start, t0 = inside.pop(i)
                trips.setdefault((start, s), []).append(t - t0)
                log.append(f"{call}    {start} -> {s} in {t - t0}")
            else:
                a, b = args
                got = us.getAverageTime(a, b)
                want = sum(trips[(a, b)]) / len(trips[(a, b)])
                log.append(f"{call} -> {got!r}   (want {want})")
                if got is None or abs(float(got) - want) > 1e-5:
                    log.append(f"   !! {call} must return {want}, got {got!r}")
                    log.append(f"      journeys on that route: {trips[(a, b)]}")
                    return False, log
        except Exception as e:
            log.append(f"   !! {call} raised {type(e).__name__}: {e}")
            return False, log

        # after EVERY call, re-check every route seen so far
        for (a, b), ds in trips.items():
            want = sum(ds) / len(ds)
            try:
                got = us.getAverageTime(a, b)
            except Exception as e:
                log.append(f"   !! after {call}, getAverageTime({a!r}, {b!r}) "
                           f"raised {type(e).__name__}: {e}")
                return False, log
            if got is None or abs(float(got) - want) > 1e-5:
                log.append(f"   !! after {call}, getAverageTime({a!r}, {b!r}) "
                           f"is {got!r}, should be {want}")
                log.append(f"      journeys on that route: {ds}")
                return False, log

    return True, log


def stress(n, seed=0, riders=6, stations=4):
    '''Many passengers around a small network, so cards are re-used constantly.'''
    random.seed(seed)
    names = [f"S{i}" for i in range(stations)]
    ops, inside, t = [], {}, 1
    for _ in range(n):
        t += random.randint(1, 20)
        rider = random.randint(1, riders)
        if rider in inside:
            dest = random.choice(names)
            ops.append(("checkOut", (rider, dest, t)))
            del inside[rider]
        else:
            src = random.choice(names)
            ops.append(("checkIn", (rider, src, t)))
            inside[rider] = src
    return check(ops)


def report(name, ok, log, tail=6):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [ ]:
# tests
LEETCODE = [
    ("checkIn",  (45, "Leyton", 3)),
    ("checkIn",  (32, "Paradise", 8)),
    ("checkIn",  (27, "Leyton", 10)),
    ("checkOut", (45, "Waterloo", 15)),
    ("checkOut", (27, "Waterloo", 20)),
    ("checkOut", (32, "Cambridge", 22)),
    ("getAverageTime", ("Paradise", "Cambridge")),
    ("getAverageTime", ("Leyton", "Waterloo")),
    ("checkIn",  (10, "Leyton", 24)),
    ("getAverageTime", ("Leyton", "Waterloo")),
    ("checkOut", (10, "Waterloo", 38)),
    ("getAverageTime", ("Leyton", "Waterloo")),
]

CASES = [
    ("the LeetCode example", LEETCODE),

    ("one journey, one average", [
        ("checkIn", (1, "A", 0)), ("checkOut", (1, "B", 5)),
        ("getAverageTime", ("A", "B"))]),

    ("question 5: the same card travels twice", [
        ("checkIn", (1, "A", 0)),   ("checkOut", (1, "B", 10)),
        ("checkIn", (1, "A", 100)), ("checkOut", (1, "B", 120)),
        ("getAverageTime", ("A", "B"))]),                       # 15.0, not 60.0

    ("question 4: A->B and B->A are different routes", [
        ("checkIn", (1, "A", 0)),  ("checkOut", (1, "B", 10)),
        ("checkIn", (2, "B", 20)), ("checkOut", (2, "A", 100)),
        ("getAverageTime", ("A", "B")), ("getAverageTime", ("B", "A"))]),

    ("question 3: station names that break string concatenation", [
        ("checkIn", (1, "Kings", 0)),      ("checkOut", (1, "Cross", 10)),
        ("checkIn", (2, "King", 0)),       ("checkOut", (2, "sCross", 30)),
        ("getAverageTime", ("Kings", "Cross")), ("getAverageTime", ("King", "sCross"))]),

    ("a passenger checks out at the station they entered", [
        ("checkIn", (1, "A", 0)), ("checkOut", (1, "A", 7)),
        ("getAverageTime", ("A", "A"))]),

    ("many passengers in flight at once", [
        ("checkIn", (i, "A", i)) for i in range(1, 11)] + [
        ("checkOut", (i, "B", i + 10)) for i in range(1, 11)] + [
        ("getAverageTime", ("A", "B"))]),

    ("averages that are not whole numbers", [
        ("checkIn", (1, "A", 0)), ("checkOut", (1, "B", 1)),
        ("checkIn", (2, "A", 0)), ("checkOut", (2, "B", 2)),
        ("getAverageTime", ("A", "B"))]),                       # 1.5

    ("interleaved journeys on overlapping routes", [
        ("checkIn", (1, "A", 0)),  ("checkIn", (2, "A", 1)),  ("checkIn", (3, "B", 2)),
        ("checkOut", (2, "C", 9)), ("checkOut", (1, "B", 10)), ("checkOut", (3, "C", 20)),
        ("getAverageTime", ("A", "B")), ("getAverageTime", ("A", "C")),
        ("getAverageTime", ("B", "C"))]),

    ("the time ceiling", [
        ("checkIn", (10**6, "A", 1)), ("checkOut", (10**6, "B", 10**6)),
        ("getAverageTime", ("A", "B"))]),
]

for name, ops in CASES:
    report(name, *check(ops))

for n, seed, riders, stations in [(50, 1, 3, 2), (200, 2, 6, 4), (1000, 3, 20, 6), (5000, 4, 50, 10)]:
    report(f"stress: {n} calls (seed {seed}, {riders} cards, {stations} stations)",
           *stress(n, seed, riders, stations))

## After it passes

- **Prove the memory claim - this is the one the tests cannot do for you.** Run the
  5000-call stress, then print `len(self.travelling)` and `len(self.routes)`. The first
  should be small: roughly the number of people mid-journey, never more than the number
  of distinct cards. The second should be at most `stations²`. If `travelling` keeps
  growing you forgot question 5's removal - and every test still passed, which is the
  entire point of doing this by hand.
- **Build the list-of-every-journey version** (question 2, option a) and run the same
  tests. It passes. Now run the 5000-call stress on both and compare the memory. Then
  answer the question that justifies it existing at all: with the list you can compute
  the **median** journey time, or the 95th percentile, and with `(total, count)` you can
  never get either back. Real transit dashboards want the 95th percentile, not the mean -
  which of your two designs would you actually deploy?
- **The invariant.** *A card id is in `travelling` if and only if that passenger has
  checked in and not yet checked out.* Write which line of `checkOut` defends it, and
  what a missing `pop` does to the very next journey by that card - the test named in
  question 5 exists for exactly this.
- **Make it real.** Add `getFare(id)` charged at check-out from a per-route price table -
  now `checkOut` needs the route *and* the duration, which you already have. Then handle
  the case a real system dreads: a passenger who checks in and **never** checks out.
  Right now they sit in your dict forever. What is the rule - a maximum journey time, a
  nightly sweep? - and which of your two dicts does the sweep touch?
- Siblings: #359 Logger Rate Limiter (a dict whose entries have a lifetime),
  #1244 Design A Leaderboard (a dict of running totals, but now you must rank them),
  #295 Find Median from Data Stream (the "you cannot get the median back from a sum"
  point, as its own problem).